# Week 4: Transfer Learning, BERT (Homework)

## Question Search Engine

Embeddings are a good source of information for solving various tasks. For example, we can classify texts or find similar documents using their representations. We already know about word2vec, GloVe and fasttext, but they don't use context information from given text (only from contexts of source data).

For today we will use full power of context-aware embeddings to find text duplicates!

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [9]:
!unzip checkpoint-22742.zip -d /


Archive:  checkpoint-22742.zip
   creating: /content/deberta/checkpoint-22742/
  inflating: /content/deberta/checkpoint-22742/config.json  
  inflating: /content/deberta/checkpoint-22742/optimizer.pt  
  inflating: /content/deberta/checkpoint-22742/rng_state.pth  
  inflating: /content/deberta/checkpoint-22742/training_args.bin  
  inflating: /content/deberta/checkpoint-22742/tokenizer.json  
  inflating: /content/deberta/checkpoint-22742/scheduler.pt  
  inflating: /content/deberta/checkpoint-22742/tokenizer_config.json  
  inflating: /content/deberta/checkpoint-22742/special_tokens_map.json  
  inflating: /content/deberta/checkpoint-22742/scaler.pt  
  inflating: /content/deberta/checkpoint-22742/spm.model  
  inflating: /content/deberta/checkpoint-22742/trainer_state.json  
 extracting: /content/deberta/checkpoint-22742/added_tokens.json  
  inflating: /content/deberta/checkpoint-22742/model.safetensors  


In [10]:
#%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets
import numpy as np

### Data Preparation

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device

device(type='cuda')

In [13]:
qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

Repo card metadata block was not found. Setting CardData to empty.




Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [15]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name).to(device)

In [16]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [ ]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [ ]:
print(repr(qqp_preprocessed["train"][0]), "...")

{'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate', 'input_ids': [101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136, 102, 5979, 1634, 1104, 3073, 20488, 2116, 1110, 1536, 1111, 1103, 12211, 179, 1233, 6451, 1571, 136, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

### Evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [ ]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=32, shuffle=False, collate_fn=transformers.default_data_collator
)

In [17]:
import tqdm

'''total, accept = 0, 0
for batch in tqdm.tqdm(val_loader):
    batch = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
      predicted = model(
          input_ids=batch["input_ids"].to(device),
          attention_mask=batch["attention_mask"].to(device),
          token_type_ids=batch["token_type_ids"].to(device),
      )

    p = torch.argmax(predicted.logits, dim=1)

    total += batch['labels'].size(0)
    accept += (p == batch['labels']).sum().item()


accuracy = accept / total

print(f'Accracy: {accuracy}')'''


'total, accept = 0, 0\nfor batch in tqdm.tqdm(val_loader):\n    batch = {k: v.to(device) for k, v in batch.items()}\n\n    with torch.no_grad():\n      predicted = model(\n          input_ids=batch["input_ids"].to(device),\n          attention_mask=batch["attention_mask"].to(device),\n          token_type_ids=batch["token_type_ids"].to(device),\n      )\n\n    p = torch.argmax(predicted.logits, dim=1)\n\n    total += batch[\'labels\'].size(0)\n    accept += (p == batch[\'labels\']).sum().item()\n\n\naccuracy = accept / total\n\nprint(f\'Accracy: {accuracy}\')'

**Task 1 (1 point)**

- Measure the validation accuracy of your model. Doing so naively may take several hours. Please make sure you use the following optimizations:
  - Run the model on GPU with no_grad
  - Using batch size larger than 1
  - Use optimize data loader with num_workers > 1
  - (Optional) Use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [ ]:
model.eval()

total, accept = 0, 0
for batch in tqdm.tqdm(val_loader):
    batch = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
      predicted = model(
          input_ids=batch["input_ids"].to(device),
          attention_mask=batch["attention_mask"].to(device),
          token_type_ids=batch["token_type_ids"].to(device),
      )

    p = torch.argmax(predicted.logits, dim=1)

    total += batch['labels'].size(0)
    accept += (p == batch['labels']).sum().item()


accuracy = accept / total

print(f'Accracy: {accuracy}')

100%|██████████| 1264/1264 [04:27<00:00,  4.73it/s]

Accracy: 0.9083848627256987


In [ ]:
assert 0.9 < accuracy < 0.91
print('Success')

Success


### Training (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

**Task 2 (4 points)**
- Choose Option A or Option B (only one will be graded)
- Follow all the instructions and restrictions

In [28]:
model = transformers.AutoModelForSequenceClassification.from_pretrained('microsoft/deberta-v3-small', num_labels=2).to(device)
tokenizer = transformers.AutoTokenizer.from_pretrained('microsoft/deberta-v3-small')

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [29]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [30]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=32, shuffle=False, collate_fn=transformers.default_data_collator
)

In [31]:
model.eval()

total, accept = 0, 0
for batch in tqdm.tqdm(val_loader):
    batch = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
      predicted = model(
          input_ids=batch["input_ids"].to(device),
          attention_mask=batch["attention_mask"].to(device),
          token_type_ids=batch["token_type_ids"].to(device),
      )

    p = torch.argmax(predicted.logits, dim=1)

    total += batch['labels'].size(0)
    accept += (p == batch['labels']).sum().item()


accuracy = accept / total

print(f'Accracy: {accuracy}')

100%|██████████| 1264/1264 [02:50<00:00,  7.41it/s]

Accracy: 0.6318327974276527


In [ ]:
model.train()

In [ ]:
from transformers import TrainingArguments, Trainer

In [ ]:
training_args = TrainingArguments(
    output_dir='./deberta',
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    report_to="none",

    save_strategy="steps",
    save_steps=1000,

    learning_rate=5e-5,
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,

    fp16=True,

    dataloader_num_workers=4,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=qqp_preprocessed["train"],
    eval_dataset=qqp_preprocessed["validation"],
    tokenizer=tokenizer,
)

/tmp/ipython-input-68310034.py:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Step,Training Loss,Validation Loss
500,0.199300,0.430198
1000,0.151700,0.394544
1500,0.303200,0.331940
2000,0.282700,0.285682
2500,0.301400,0.272581
3000,0.270500,0.266658
3500,0.258800,0.264992
4000,0.279000,0.259845
4500,0.266000,0.270288
5000,0.264800,0.248988


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: T

TrainOutput(global_step=22742, training_loss=0.20474256882932987, metrics={'train_runtime': 7569.0413, 'train_samples_per_second': 96.141, 'train_steps_per_second': 3.005, 'total_flos': 2.4099724986107904e+16, 'train_loss': 0.20474256882932987, 'epoch': 2.0})

In [32]:
model_dir = "/content/deberta/checkpoint-22742"
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
tokenizer = transformers.AutoTokenizer.from_pretrained(model_dir)

In [33]:
model.eval()

total, accept = 0, 0
for batch in tqdm.tqdm(val_loader):
    batch = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
      predicted = model(
          input_ids=batch["input_ids"].to(device),
          attention_mask=batch["attention_mask"].to(device),
          token_type_ids=batch["token_type_ids"].to(device),
      )

    p = torch.argmax(predicted.logits, dim=1)

    total += batch['labels'].size(0)
    accept += (p == batch['labels']).sum().item()


accuracy = accept / total
print()
print(f'Accracy: {accuracy}')



100%|██████████| 1264/1264 [02:51<00:00,  7.39it/s]


Accracy: 0.9170418006430868


### Finding Duplicates (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

**Task 3 (1 point)**
- Implement function for finding duplicates
- Test it on several examples (at least 5)
- Check suggested duplicates and make a conclusion about model correctness

In [131]:
unic_text_from_dataset= set()
for example in tqdm.tqdm(qqp_preprocessed['train']):
  unic_text_from_dataset.add(example['text1'])
  unic_text_from_dataset.add(example['text2'])

unic_text_from_dataset = list(unic_text_from_dataset)

100%|██████████| 363846/363846 [01:35<00:00, 3802.61it/s]


In [141]:
import tqdm
def finding_diplicates(model, tokenizer, questions, dataset, top_n=5, batch_size=64):
  model.eval()


  scores = {q: [] for q in questions}

  for q in questions:
    text_to_compare = [t for t in unic_text_from_dataset if t != q]
    all_scores = []

    with tqdm.tqdm(
        total=len(text_to_compare),
        leave=False
    ) as pbar:

      for i in range(0, len(text_to_compare), batch_size):
        batch_text = text_to_compare[i:i + batch_size]
        inputs = tokenizer(
            [q] * len(batch_text),
            batch_text,
            padding="max_length",
            max_length=MAX_LENGTH,
            truncation=True,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
          outputs = model(**inputs)
          p = F.softmax(outputs.logits, dim=1)[:, 1]

        for score, text in zip(p.cpu().tolist(), batch_text):
          if (score > 0.5):
            all_scores.append((score, text))

        pbar.update(len(batch_text))

    scores[q] = sorted(all_scores, key=lambda x: x[0], reverse=True)[:top_n]

  return scores


In [133]:
qqp_preprocessed['train'][1]['text1']

'How do I control my horny emotions?'

In [134]:
test_q = []

for i in qqp_preprocessed['train'].select(range(5)):
  test_q.append(i['text1'])

test_q


['How is the life of a math student? Could you describe your own experiences?',
 'How do I control my horny emotions?',
 'What causes stool color to change to yellow?',
 'What can one do after MBBS?',
 'Where can I find a power outlet for my laptop at Melbourne Airport?']

In [135]:
test = [
    ('What is AI?', 'What is artificial intelligence?'),
    ('How to learn Python?', "What's the best way to learn Python programming?"),
    ('How to make delicious pizza?', 'What is the main problem with Google marketing?'),
    ('How to study a lot and not get tired?', 'How to control your time and energy?'),
    ("Why don't people simply 'Google' instead of asking questions on Quora?", "Why do people ask Quora questions instead of just searching google?")
]

for q1, q2 in test:
  inputs = tokenizer(q1, q2, return_tensors="pt", padding=True, truncation=True).to(device)
  with torch.no_grad():
      outputs = model(**inputs)
      p = F.softmax(outputs.logits, dim=1)[0][1].item()
  print(f"'{q1}' and '{q2}': {p}")

'What is AI?' and 'What is artificial intelligence?': 0.000509448756929487
'How to learn Python?' and 'What's the best way to learn Python programming?': 0.23870640993118286
'How to make delicious pizza?' and 'What is the main problem with Google marketing?': 2.136676266673021e-05
'How to study a lot and not get tired?' and 'How to control your time and energy?': 0.0011049848981201649
'Why don't people simply 'Google' instead of asking questions on Quora?' and 'Why do people ask Quora questions instead of just searching google?': 0.9977834820747375


In [142]:
test_dataset = qqp_preprocessed['train']

res = finding_diplicates(model, tokenizer, test_q, test_dataset)
print()
for q, duplicates in res.items():
    print(f'Now question: {q}')
    if (len(duplicates) == 0):
       print('Nothing to show(')
    else:
      t = 1;
      for r in duplicates:
        print(f'{t}: {r[1]}; score: {r[0]}')
        t += 1
    print('-' * 150)



KeyboardInterrupt: 

### Bonus: Finding Duplicates Faster (0.5 point)

Try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

**Bonus Task 1 (0.5 point)**
- Speed up your implementation from "Finding Duplicates" part
- Capture both old and new implementation work time
- Describe your approach

In [ ]:
<A whole lot of YOUR CODE HERE>

### Bonus: Finding Duplicates in Old-Fashioned way (1.5 points)

In this bonus task you are supposed to use pretrained embeddings (word2vec, GloVe or fasttext) for solving the duplicates problem.

**Bonus Task 2 (1.5 points)**
- Solve Finding Duplicates problem using mentioned embeddings
- Compare old-fashioned solution to previous ones (quality, speed, etc.)
- Make a small report (up to 5 steps, results and conclusions) on work done in this part

In [ ]:
<A whole lot of YOUR CODE HERE>